In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-30

@author:       Oscar Trevizo
@institution:  Harvard Extension School
@credential:   Graduate Certificate in Data Science (2023)
@context:      Independent project — applying graduate-level concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

Agentic Model Reasoning Divergence
====================================

Hypothesis:
    LLM agents diverge in reasoning strategy on questions that require
    filtering by one metric and ranking by another simultaneously
    (cross-metric reasoning). This divergence is observable in the tool
    call sequence — specifically in the top_n parameter chosen for the
    secondary query — and may produce answers of different scope even
    when the final answer (top country) agrees.

Experiment:
    Same question asked to three models using identical tools, data,
    and PRODUCT_REGISTRY. Tool call sequences and answers are captured
    and compared.

    Question: "Which country in the top 20 GDP has the highest
               migration rate?"

    Models tested:
        1. OpenAI   gpt-4o-mini
        2. Anthropic claude-haiku-4-5-20251001
        3. Anthropic claude-sonnet-4-6

Neurosymbolic framing:
    The symbolic scaffolding (DataProduct tools, PRODUCT_REGISTRY,
    data pipeline) is identical and deterministic across all runs.
    All observed variance is attributable to the neural component
    (LLM reasoning). This isolates the neural contribution to outcome
    divergence in a hybrid system.

Support module : ../python_vignettes/data_products/data_product_lib.py
Pattern follows: machine_learning/agentic_ai_vignette_yfinance.ipynb
                 machine_learning/ai_hello_world.ipynb

Revision History:
    2026-06-30  Original development
                - Three-model experiment: GPT-4o-mini, Haiku, Sonnet
                - Provider-agnostic run_agent() with schema converter
                - Structured results capture and comparison table
"""

'\nCreated on 2026-06-30\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School\n@credential:   Graduate Certificate in Data Science (2023)\n@context:      Independent project — applying graduate-level concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nAgentic Model Reasoning Divergence\n====================================\n\nHypothesis:\n    LLM agents diverge in reasoning strategy on questions that require\n    filtering by one metric and ranking by another simultaneously\n    (cross-metric reasoning). This divergence is observable in the tool\n    call sequence — specifically in the top_n parameter chosen for the\n    secondary query — and may produce answers of different scope even\n    when the final answer (top country) agrees.\n\nExperiment:\n    Same question asked to three models using identical tools, data,\n    and PRODUCT_REGISTRY. Tool call sequences and answers are captured\n    and compared.\n\n    Question: "Which co

## Hypothesis

> **LLM agents diverge on cross-metric reasoning questions** — specifically
> questions that require (1) filtering a dataset by one metric and
> (2) ranking the filtered result by a second metric simultaneously.

This divergence is not visible in the final answer (all correct models
return Canada) but in the **tool call strategy** — particularly the
`top_n` parameter chosen for the migration rate query:

| Strategy | top_n | What it answers |
|---|---|---|
| **Intersection** | 20 | Countries in global top-20 by *both* GDP and migration |
| **Filter-then-rank** | 100-210 | Top-20 GDP countries ranked by migration within that set |

The two strategies answer subtly different questions. The intersection
strategy may miss countries like Netherlands (#18 GDP, #3 migration)
that do not appear in the global top-20 by migration rate alone.

### Neurosymbolic framing

The **symbolic side** (DataProduct tools, PRODUCT_REGISTRY, data pipeline)
is fixed and identical across all three model runs. All observed variance
comes from the **neural side** (LLM reasoning). This is a controlled
experiment isolating neural contribution to outcome divergence.

In [2]:
import io
import json
import os
import sys
from datetime import datetime, timezone

import anthropic
import numpy as np
import pandas as pd
from openai import OpenAI

# Import data_product_lib from the data_products vignette folder
sys.path.insert(0, '../python_vignettes/data_products/')
from data_product_lib import (
    DataProductMetadata,
    SemanticLayer,
    LineageTracker,
    DataProduct,
)

anthropic_client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
openai_client    = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

print(f"Anthropic SDK : {anthropic.__version__}")
import openai as _oai
print(f"OpenAI SDK    : {_oai.__version__}")

Anthropic SDK : 0.88.0
OpenAI SDK    : 2.30.0


## Experimental Design

| Element | Value |
|---|---|
| **Question** | *"Which country in the top 20 GDP has the highest migration rate?"* |
| **Phrasing** | Original vague phrasing — no explicit two-step instruction |
| **Data** | UN WPP 2024 (demographics) + World Bank WDI (GDP) |
| **Tools** | 8 tools (same as `agentic_data_product_vignette.ipynb`) |
| **Models** | GPT-4o-mini · Haiku · Sonnet |
| **Runs per model** | 1 (extend to N for variance study) |
| **Controlled** | Symbolic scaffolding — identical tools, data, registry |
| **Variable** | LLM reasoning — model only |

### What we measure

- **Tool call sequence** — which tools, in what order
- **Migration `top_n`** — 20 (intersection) vs 100+ (filter-then-rank)
- **Answer** — which country named as winner
- **Correct?** — Canada is the verified answer (11.039 per 1,000, 2023)
- **Iterations** — number of API round-trips

In [3]:
# Pipeline functions — same as agentic_data_product_vignette.ipynb.
# data_product_lib is imported via sys.path; pipeline functions are
# reproduced here to keep the notebook self-contained.

UN_FILE            = '../data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT_20260327.xlsx'
WB_GDP_FILE        = '../data/WB_GDP.xls'
WB_GDP_GROWTH_FILE = '../data/WB_GDP_growth.xls'


def load_un_wpp(filepath, year_start=1961, year_end=2023):
    df = pd.read_excel(
        filepath, sheet_name='Estimates', skiprows=16, thousands=' ',
        usecols=[
            'Region, subregion, country or area *', 'ISO3 Alpha-code',
            'ISO2 Alpha-code', 'Type', 'Year',
            'Total Population, as of 1 July (thousands)',
            'Median Age, as of 1 July (years)',
            'Population Growth Rate (percentage)',
            'Total Fertility Rate (live births per woman)',
            'Life Expectancy at Birth, both sexes (years)',
            'Net Number of Migrants (thousands)',
            'Net Migration Rate (per 1,000 population)',
        ]
    )
    df = df.rename(columns={
        'Region, subregion, country or area *'         : 'Location',
        'ISO3 Alpha-code'                              : 'ISO3',
        'ISO2 Alpha-code'                              : 'ISO2',
        'Type'                                         : 'LocType',
        'Total Population, as of 1 July (thousands)'  : 'Population_Ks',
        'Median Age, as of 1 July (years)'             : 'MedAge',
        'Population Growth Rate (percentage)'          : 'PopulationGrowthRate',
        'Total Fertility Rate (live births per woman)' : 'FertilityRate_births_per_woman',
        'Life Expectancy at Birth, both sexes (years)' : 'LifeExpectancy',
        'Net Number of Migrants (thousands)'           : 'NetMigrants_Ks',
        'Net Migration Rate (per 1,000 population)'    : 'NetMigrationRate_per_Kpop',
    })
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(np.int64)
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    df.loc[df['LocType'] == 'Country/Area', 'LocType'] = 'Country'
    df['ReceivesMigrants']    = (df['NetMigrants_Ks'] > 0).astype(int)
    df['ImmigrantsEmigrants'] = df['NetMigrants_Ks'].apply(
        lambda x: 'Immigrants' if x > 0 else 'Emigrants')
    return df.reset_index(drop=True)


def filter_countries(df):
    return df[df['LocType'] == 'Country'].reset_index(drop=True)


def clean_types(df):
    for col in ['Population_Ks', 'MedAge', 'PopulationGrowthRate',
                'FertilityRate_births_per_woman', 'LifeExpectancy',
                'NetMigrants_Ks', 'NetMigrationRate_per_Kpop']:
        df = df.copy()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_wb_gdp(gdp_file, gdp_growth_file, year_start=1961, year_end=2023):
    def _load_one(filepath, indicator):
        df = pd.read_excel(filepath, skiprows=3, sheet_name='Data')
        df = df.rename(columns={'Country Code': 'ISO3'})
        df = df.drop(columns=['Country Name', 'Indicator Name', 'Indicator Code'])
        df = df.drop(columns=[c for c in df.columns if str(c) == '2025'], errors='ignore')
        df = df.set_index('ISO3').stack().reset_index()
        df.columns = ['ISO3', 'Year', indicator]
        df['Year']    = df['Year'].astype(np.int64)
        df[indicator] = pd.to_numeric(df[indicator], errors='coerce')
        return df
    df = pd.merge(_load_one(gdp_file, 'GDP_USD'),
                  _load_one(gdp_growth_file, 'GDP_growth_pct'),
                  on=['ISO3', 'Year'])
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    return df.reset_index(drop=True)


def _merge_data_products(dp1, dp2, on):
    collisions = set(dp1.semantic_layer.to_dict()) & set(dp2.semantic_layer.to_dict())
    if collisions:
        raise ValueError(f'Semantic name collision(s): {sorted(collisions)}.')
    merged_df = pd.merge(dp1.data, dp2.data, on=on, how='inner')
    merged_sl = SemanticLayer()
    for name, e in dp1.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    for name, e in dp2.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    merged_lin = LineageTracker()
    for s in dp1.lineage.to_list():
        merged_lin.log(f"dp1/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    for s in dp2.lineage.to_list():
        merged_lin.log(f"dp2/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    merged_lin.log('merge', 'inner_join', dp1.data.shape, merged_df.shape,
                   f'keys: {"+".join(on)}')
    return merged_df, merged_sl, merged_lin


print("Pipeline functions ready.")

Pipeline functions ready.


In [4]:
PRODUCT_REGISTRY: dict = {}

SOURCE_CONFIGS = {
    "UN_WPP":         {"description": "UN World Population Prospects 2024"},
    "WORLD_BANK_GDP": {"description": "World Bank WDI — GDP indicators"},
}


def list_available_sources() -> dict:
    return {"available_sources": SOURCE_CONFIGS,
            "loaded": list(PRODUCT_REGISTRY.keys())}


def load_source(source_name: str,
                year_start: int = 1961, year_end: int = 2023) -> dict:
    if source_name in PRODUCT_REGISTRY:
        dp = PRODUCT_REGISTRY[source_name]
        return {"status": "already_loaded", "product_name": source_name,
                "shape": list(dp.data.shape)}
    if source_name == "UN_WPP":
        lin = LineageTracker()
        raw  = load_un_wpp(UN_FILE, year_start, year_end)
        lin.log("1-load",   "load_un_wpp",        (0, 0),    raw.shape,  f"{year_start}-{year_end}")
        ctry = filter_countries(raw)
        lin.log("2-filter", "filter_countries",    raw.shape, ctry.shape, "LocType==Country")
        df   = clean_types(ctry)
        lin.log("3-clean",  "clean_types",          ctry.shape, df.shape, "to_numeric")
        sl = SemanticLayer()
        sl.register("net_migration_rate", "NetMigrationRate_per_Kpop", "per 1,000 population",
                    "Net migrants per 1,000 residents", source_system="UN WPP 2024")
        sl.register("net_migrants", "NetMigrants_Ks", "thousands",
                    "Net number of migrants", source_system="UN WPP 2024")
        sl.register("population", "Population_Ks", "thousands",
                    "Total population, as of 1 July", source_system="UN WPP 2024")
        sl.register("fertility_rate", "FertilityRate_births_per_woman", "births per woman",
                    "Total fertility rate", source_system="UN WPP 2024")
        sl.register("life_expectancy", "LifeExpectancy", "years",
                    "Life expectancy at birth", source_system="UN WPP 2024")
        meta = DataProductMetadata(
            name="UN WPP Demographics", description="Country-year panel, UN WPP 2024",
            domain="Demographics", owner="Experiment",
            source="UN World Population Prospects 2024",
            source_url="https://population.un.org/wpp/",
            license="CC BY 3.0 IGO", version="1.0",
            refresh_frequency="Every 2 years",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)
    elif source_name == "WORLD_BANK_GDP":
        lin = LineageTracker()
        df  = load_wb_gdp(WB_GDP_FILE, WB_GDP_GROWTH_FILE, year_start, year_end)
        lin.log("1-load", "load_wb_gdp", (0, 0), df.shape, f"{year_start}-{year_end}")
        sl = SemanticLayer()
        sl.register("gdp", "GDP_USD", "constant USD",
                    "GDP (NY.GDP.MKTP.KD)", source_system="World Bank WDI")
        sl.register("gdp_growth", "GDP_growth_pct", "% per year",
                    "GDP growth (NY.GDP.MKTP.KD.ZG)", source_system="World Bank WDI")
        meta = DataProductMetadata(
            name="World Bank GDP", description="Country-year panel, World Bank WDI",
            domain="Economics", owner="Experiment",
            source="World Bank World Development Indicators",
            source_url="https://databank.worldbank.org/",
            license="CC BY 4.0", version="1.0",
            refresh_frequency="Annual",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)
    else:
        return {"error": f"Unknown source '{source_name}'."}
    PRODUCT_REGISTRY[source_name] = dp
    return {"status": "loaded", "product_name": source_name,
            "shape": list(dp.data.shape),
            "semantic_names": list(dp.semantic_layer.to_dict().keys())}


def check_semantic_conflicts(source1_name, source2_name):
    missing = [n for n in (source1_name, source2_name) if n not in PRODUCT_REGISTRY]
    if missing:
        return {"error": f"Not in registry: {missing}"}
    n1 = set(PRODUCT_REGISTRY[source1_name].semantic_layer.to_dict())
    n2 = set(PRODUCT_REGISTRY[source2_name].semantic_layer.to_dict())
    cols = sorted(n1 & n2)
    return {"collisions": cols, "safe_to_merge": len(cols) == 0,
            "source1_names": sorted(n1), "source2_names": sorted(n2)}


def rename_semantic_entry(product_name, old_name, new_name):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    sl = PRODUCT_REGISTRY[product_name].semantic_layer
    if old_name not in sl._registry:
        return {"error": f"'{old_name}' not found"}
    e = sl._registry[old_name]
    sl.register(new_name, e["column"], e["unit"], e["description"], e["source_system"])
    del sl._registry[old_name]
    return {"status": "renamed", "old_name": old_name, "new_name": new_name}


def merge_sources(source1_name, source2_name, product_name="MERGED"):
    missing = [n for n in (source1_name, source2_name) if n not in PRODUCT_REGISTRY]
    if missing:
        return {"error": f"Not in registry: {missing}"}
    dp1, dp2 = PRODUCT_REGISTRY[source1_name], PRODUCT_REGISTRY[source2_name]
    try:
        mdf, msl, mlin = _merge_data_products(dp1, dp2, on=["ISO3", "Year"])
    except ValueError as exc:
        return {"error": str(exc)}
    meta = DataProductMetadata(
        name=product_name, description=f"Merged: {source1_name}+{source2_name}",
        domain="Multi-domain", owner="Experiment",
        source=f"{dp1.metadata.source}+{dp2.metadata.source}",
        source_url="", license="", version="1.0",
        refresh_frequency="On demand",
        created_at=datetime.now(timezone.utc).isoformat())
    dp = DataProduct(mdf, meta, msl, mlin)
    PRODUCT_REGISTRY[product_name] = dp
    return {"status": "merged", "product_name": product_name,
            "shape": list(dp.data.shape),
            "semantic_names": sorted(dp.semantic_layer.to_dict().keys())}


def query_product(product_name, business_name, year=None, top_n=10):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp  = PRODUCT_REGISTRY[product_name]
    sl  = dp.semantic_layer.to_dict()
    if business_name not in sl:
        return {"error": f"'{business_name}' not in semantic layer",
                "available": sorted(sl.keys())}
    entry  = sl[business_name]
    col    = entry["column"]
    cols   = [c for c in ["Location", "ISO3", "Year", col] if c in dp.data.columns]
    df     = dp.data[cols].copy()
    if year is not None:
        df = df[df["Year"] == year]
    result = df.nlargest(top_n, col).reset_index(drop=True)
    return {"business_name": business_name, "resolved_column": col,
            "unit": entry["unit"], "year_filter": year, "top_n": top_n,
            "results": result.to_dict(orient="records")}


def get_product_card(product_name):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    buf = io.StringIO()
    import sys as _sys
    old, _sys.stdout = _sys.stdout, buf
    PRODUCT_REGISTRY[product_name].card()
    _sys.stdout = old
    return {"product_name": product_name, "card": buf.getvalue()}


def get_lineage(product_name):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp = PRODUCT_REGISTRY[product_name]
    return {"product_name": product_name, "steps": dp.lineage.to_list()}


TOOL_FUNCTIONS = {
    "list_available_sources":   list_available_sources,
    "load_source":              load_source,
    "check_semantic_conflicts": check_semantic_conflicts,
    "rename_semantic_entry":    rename_semantic_entry,
    "merge_sources":            merge_sources,
    "query_product":            query_product,
    "get_product_card":         get_product_card,
    "get_lineage":              get_lineage,
}

print(f"PRODUCT_REGISTRY + {len(TOOL_FUNCTIONS)} tools ready.")

PRODUCT_REGISTRY + 8 tools ready.


In [5]:
# Anthropic tool schemas (input_schema format)
TOOLS_ANTHROPIC = [
    {"name": "list_available_sources",
     "description": "Return available data sources and which are loaded.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "load_source",
     "description": "Load a source ('UN_WPP' or 'WORLD_BANK_GDP') and register it.",
     "input_schema": {"type": "object",
       "properties": {
         "source_name": {"type": "string"},
         "year_start":  {"type": "integer"},
         "year_end":    {"type": "integer"}},
       "required": ["source_name"]}},
    {"name": "check_semantic_conflicts",
     "description": "Check for business-name collisions before merging two sources.",
     "input_schema": {"type": "object",
       "properties": {
         "source1_name": {"type": "string"},
         "source2_name": {"type": "string"}},
       "required": ["source1_name", "source2_name"]}},
    {"name": "rename_semantic_entry",
     "description": "Rename a business name in a product's semantic layer to resolve a conflict.",
     "input_schema": {"type": "object",
       "properties": {
         "product_name": {"type": "string"},
         "old_name":     {"type": "string"},
         "new_name":     {"type": "string"}},
       "required": ["product_name", "old_name", "new_name"]}},
    {"name": "merge_sources",
     "description": "Inner join two DataProducts on ISO3+Year.",
     "input_schema": {"type": "object",
       "properties": {
         "source1_name": {"type": "string"},
         "source2_name": {"type": "string"},
         "product_name": {"type": "string"}},
       "required": ["source1_name", "source2_name"]}},
    {"name": "query_product",
     "description": "Query a registered product by semantic business name. Returns top-N rows.",
     "input_schema": {"type": "object",
       "properties": {
         "product_name":  {"type": "string"},
         "business_name": {"type": "string"},
         "year":          {"type": "integer"},
         "top_n":         {"type": "integer"}},
       "required": ["product_name", "business_name"]}},
    {"name": "get_product_card",
     "description": "Return the card summary of a registered data product.",
     "input_schema": {"type": "object",
       "properties": {"product_name": {"type": "string"}},
       "required": ["product_name"]}},
    {"name": "get_lineage",
     "description": "Return the lineage audit trail of a registered data product.",
     "input_schema": {"type": "object",
       "properties": {"product_name": {"type": "string"}},
       "required": ["product_name"]}},
]


def to_openai_tools(anthropic_tools):
    # Anthropic uses 'input_schema'; OpenAI wraps the schema as 'parameters' inside 'function'
    return [
        {"type": "function",
         "function": {
             "name":        t["name"],
             "description": t["description"],
             "parameters":  t["input_schema"],
         }}
        for t in anthropic_tools
    ]

TOOLS_OPENAI = to_openai_tools(TOOLS_ANTHROPIC)

print(f"Anthropic schemas : {len(TOOLS_ANTHROPIC)} tools")
print(f"OpenAI schemas    : {len(TOOLS_OPENAI)} tools (auto-converted)")

Anthropic schemas : 8 tools
OpenAI schemas    : 8 tools (auto-converted)


In [6]:
SYSTEM_PROMPT = (
    "You are a Data Product Engineer agent. You compose governed data products "
    "from available sources using the tools provided.\n\n"
    "When given a request:\n"
    "1. Call list_available_sources() to see what is available.\n"
    "2. Call load_source() for each source you need.\n"
    "3. Call check_semantic_conflicts() before merging.\n"
    "4. Call merge_sources() to compose the product.\n"
    "5. Answer the user's question with query_product().\n\n"
    "Be concise. Report the country name and migration rate in your final answer."
)

MAX_ITERATIONS = 20


def run_agent(question, provider, model, verbose=True):
    # Provider-agnostic agent loop.
    # provider: 'anthropic' or 'openai'
    # Returns dict: answer, tool_calls, iterations, model, provider
    PRODUCT_REGISTRY.clear()
    tool_calls_log = []

    # Anthropic: system is a separate parameter; messages list starts with user turn.
    # OpenAI: system is role='system' first message in the list.
    if provider == "anthropic":
        messages = [{"role": "user", "content": question}]
    else:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ]

    for iteration in range(MAX_ITERATIONS):

        # ── Anthropic ─────────────────────────────────────────────────────────
        if provider == "anthropic":
            response = anthropic_client.messages.create(
                model=model, max_tokens=2048,
                system=SYSTEM_PROMPT,
                tools=TOOLS_ANTHROPIC,
                messages=messages)

            if response.stop_reason == "end_turn":
                text = next(
                    (b.text for b in response.content if hasattr(b, "text")), "")
                return {"answer": text, "tool_calls": tool_calls_log,
                        "iterations": iteration + 1,
                        "model": model, "provider": provider}

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                results = []
                for block in response.content:
                    if block.type == "tool_use":
                        entry = {
                            "name": block.name,
                            "input": block.input,
                            "migration_top_n": (
                                block.input.get("top_n")
                                if block.name == "query_product"
                                and block.input.get("business_name") == "net_migration_rate"
                                else None
                            ),
                        }
                        tool_calls_log.append(entry)
                        if verbose:
                            print(f"  -> {block.name}({block.input})")
                        fn     = TOOL_FUNCTIONS.get(block.name)
                        result = fn(**block.input) if fn else {"error": "unknown tool"}
                        results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result),
                        })
                messages.append({"role": "user", "content": results})

        # ── OpenAI ────────────────────────────────────────────────────────────
        elif provider == "openai":
            response = openai_client.chat.completions.create(
                model=model, max_tokens=2048,
                tools=TOOLS_OPENAI,
                messages=messages)

            choice = response.choices[0]

            if choice.finish_reason == "stop":
                return {"answer": choice.message.content,
                        "tool_calls": tool_calls_log,
                        "iterations": iteration + 1,
                        "model": model, "provider": provider}

            if choice.finish_reason == "tool_calls":
                messages.append(choice.message)
                for tc in choice.message.tool_calls:
                    args  = json.loads(tc.function.arguments)
                    entry = {
                        "name": tc.function.name,
                        "input": args,
                        "migration_top_n": (
                            args.get("top_n")
                            if tc.function.name == "query_product"
                            and args.get("business_name") == "net_migration_rate"
                            else None
                        ),
                    }
                    tool_calls_log.append(entry)
                    if verbose:
                        print(f"  -> {tc.function.name}({args})")
                    fn     = TOOL_FUNCTIONS.get(tc.function.name)
                    result = fn(**args) if fn else {"error": "unknown tool"}
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": json.dumps(result),
                    })

    return {"answer": "Max iterations reached.",
            "tool_calls": tool_calls_log, "iterations": MAX_ITERATIONS,
            "model": model, "provider": provider}


print("run_agent() ready — supports 'anthropic' and 'openai' providers.")

run_agent() ready — supports 'anthropic' and 'openai' providers.


---
## Experiment — Run All Three Models

The same question is sent to each model in sequence.
`PRODUCT_REGISTRY` is cleared before each run so every model
starts from an identical blank state.

In [7]:
QUESTION = (
    "Which country in the top 20 GDP has the highest migration rate?"
)

MODELS = [
    {"provider": "openai",    "model": "gpt-4o-mini",               "label": "OpenAI GPT-4o-mini"},
    {"provider": "anthropic", "model": "claude-haiku-4-5-20251001",  "label": "Anthropic Haiku"},
    {"provider": "anthropic", "model": "claude-sonnet-4-6",          "label": "Anthropic Sonnet"},
]

results = []

for cfg in MODELS:
    print(f"\n{'='*60}")
    print(f"Model : {cfg['label']}")
    print(f"Q     : {QUESTION}")
    print(f"{'='*60}")
    result = run_agent(QUESTION, cfg["provider"], cfg["model"], verbose=True)
    result["label"] = cfg["label"]
    results.append(result)
    print(f"\nAnswer:\n{result['answer']}")


Model : OpenAI GPT-4o-mini
Q     : Which country in the top 20 GDP has the highest migration rate?
  -> list_available_sources({})
  -> load_source({'source_name': 'UN_WPP'})
  -> load_source({'source_name': 'WORLD_BANK_GDP'})
  -> check_semantic_conflicts({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP'})
  -> merge_sources({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP', 'product_name': 'GDP_Migration_Data'})
  -> query_product({'product_name': 'GDP_Migration_Data', 'business_name': 'net_migration_rate', 'top_n': 20})
  -> query_product({'product_name': 'GDP_Migration_Data', 'business_name': 'gdp', 'top_n': 20})

Answer:
The country in the top 20 GDP with the highest migration rate is **China, Hong Kong SAR**, with a migration rate of **357.14 per 1,000 population**.

Model : Anthropic Haiku
Q     : Which country in the top 20 GDP has the highest migration rate?
  -> list_available_sources({})
  -> load_source({'source_name': 'WORLD_BANK_GDP'})
  -> load_sourc

---
## Results

In [8]:
def summarise_tool_calls(tool_calls):
    # Return compact string of tool names for display
    return ' -> '.join(tc['name'].replace('_', ' ') for tc in tool_calls)

def get_migration_top_n(tool_calls):
    # Extract the top_n used for the net_migration_rate query
    for tc in tool_calls:
        if tc.get('migration_top_n') is not None:
            return tc['migration_top_n']
    return None

def infer_strategy(top_n):
    if top_n is None:
        return 'unknown'
    if top_n <= 20:
        return f'intersection (top_n={top_n})'
    return f'filter-then-rank (top_n={top_n})'

rows = []
for r in results:
    top_n    = get_migration_top_n(r['tool_calls'])
    strategy = infer_strategy(top_n)
    correct  = 'canada' in r['answer'].lower()
    rows.append({
        'Model':           r['label'],
        'Strategy':        strategy,
        'Iterations':      r['iterations'],
        'Tool calls':      len(r['tool_calls']),
        'Canada correct?': 'YES' if correct else 'NO',
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

             Model                     Strategy  Iterations  Tool calls Canada correct?
OpenAI GPT-4o-mini      intersection (top_n=20)           7           7              NO
   Anthropic Haiku filter-then-rank (top_n=100)           7           7             YES
  Anthropic Sonnet      intersection (top_n=20)           7           9             YES


In [9]:
# Print full tool-call traces for each model
for r in results:
    print(f"\n{'='*60}")
    print(f"Model: {r['label']}")
    print(f"{'='*60}")
    for i, tc in enumerate(r['tool_calls'], 1):
        top_n_note = (f"  [migration_top_n={tc['migration_top_n']}]"
                      if tc['migration_top_n'] else "")
        print(f"  {i:2d}. {tc['name']}({tc['input']}){top_n_note}")


Model: OpenAI GPT-4o-mini
   1. list_available_sources({})
   2. load_source({'source_name': 'UN_WPP'})
   3. load_source({'source_name': 'WORLD_BANK_GDP'})
   4. check_semantic_conflicts({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP'})
   5. merge_sources({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP', 'product_name': 'GDP_Migration_Data'})
   6. query_product({'product_name': 'GDP_Migration_Data', 'business_name': 'net_migration_rate', 'top_n': 20})  [migration_top_n=20]
   7. query_product({'product_name': 'GDP_Migration_Data', 'business_name': 'gdp', 'top_n': 20})

Model: Anthropic Haiku
   1. list_available_sources({})
   2. load_source({'source_name': 'WORLD_BANK_GDP'})
   3. load_source({'source_name': 'UN_WPP'})
   4. check_semantic_conflicts({'source1_name': 'WORLD_BANK_GDP', 'source2_name': 'UN_WPP'})
   5. merge_sources({'source1_name': 'WORLD_BANK_GDP', 'source2_name': 'UN_WPP', 'product_name': 'GDP_MIGRATION'})
   6. query_product({'product_name'

---
## Findings (observed 2026-06-30)

| Model | Query order | Migration top_n | Year filter | Answer | Correct? |
|---|---|---|---|---|---|
| GPT-4o-mini | migration FIRST, then GDP | 20 | none | China, Hong Kong SAR (357.14) | NO |
| Haiku | GDP first, then migration | 100 | none | Confused — guessed Canada | PARTIAL |
| Sonnet | GDP first, migration, then added year=2023 to both | 20 | 2023 | Canada (+11.04) | YES |

### What happened — model by model

**GPT-4o-mini** made a query order error: it called `query(net_migration_rate, top_n=20)` first,
then `query(gdp, top_n=20)`. Without filtering by year, the global migration top-20 spans all
years 1961–2023. Hong Kong had extreme outlier migration in certain years, so it topped the list
at 357.14 — a value that is real in the data but belongs to a different historical context.
The model then reported that as the answer without cross-referencing whether Hong Kong SAR
is in the GDP top 20. It is not.

**Haiku** used the correct query order (GDP first, then migration with top_n=100 for a wider net)
but could not cross-reference the two result sets in reasoning. Without a dedicated
"filter migration by GDP rank" tool, the agent received two separate lists and had to
reason across them in text — which it failed to do systematically. It acknowledged
the limitation mid-answer, listed many candidates, and hedged on Canada as a guess
without year-specific data. The answer is partially correct by intuition, not by data.

**Sonnet** queried GDP top 20, then migration top 20 — then added a `year=2023` filter to both
queries as a third step. The temporal disambiguation reduced the search space enough
that the cross-metric intersection became tractable. It correctly identified Canada
at +11.04 per 1,000 in 2023, with a ranked comparison table.

### The structural gap — no cross-filter tool

None of the three models had a tool to answer the question directly:
*"Return migration rate, sorted descending, filtered to only the top-20 GDP countries."*

Every model had to bridge two separate `query_product()` result sets in its own reasoning.
That bridge is where divergence enters. The symbolic scaffolding returns correct data;
the neural reasoning decides how to join it — and that decision varied by model,
producing different query orders, different scope choices, and different correctness.

### Hypothesis verdict

**Supported.** All three models diverged on this cross-metric question:

- Query order differed (migration-first vs GDP-first)
- Scope strategy differed (top_n=20 intersection vs top_n=100 wide net)
- Correctness differed (wrong / partial / correct)
- Number of tool calls differed (7 / 7 / 9)

The divergence is not random noise — each failure mode is traceable to a specific
reasoning decision at the cross-metric gap: GPT-4o-mini got the order wrong;
Haiku got the order right but failed the cross-reference; Sonnet added a temporal
dimension to make the intersection tractable.

### Connection to neurosymbolic AI

The symbolic scaffolding (tools, PRODUCT_REGISTRY, DataProduct pipeline) returned
identical, correct data to all three models. The tool call traces are auditable.
The lineage is tracked. None of that helped GPT-4o-mini or Haiku reach the correct answer.

The neural component — LLM reasoning — determined the outcome. This confirms the
neurosymbolic governance hypothesis: governed structure is necessary but not sufficient.
The neural reasoning layer must also be capable of cross-metric inference. When it is not,
governance preserves auditability but cannot prevent an incorrect answer.

> A governed wrong answer is still a wrong answer.
> The symbolic layer makes it traceable. The neural layer makes it true — or not.

---
## References

- **Data:** UN WPP 2024 (population.un.org/wpp) · World Bank WDI (databank.worldbank.org)
- **Anthropic Tool Use:** docs.anthropic.com/en/docs/tool-use
- **OpenAI Function Calling:** platform.openai.com/docs/guides/function-calling
- **Data products:** Dehghani (2022) *Data Mesh*, O'Reilly
- **Support module:** `../python_vignettes/data_products/data_product_lib.py`
- **Related notebooks:** `agentic_data_product_vignette.ipynb` · `ai_hello_world.ipynb`